# 第3章 主成分分析の理論 ― デモノートブック

この章の主張のうち、**数値で確かめられるものを一つずつ実行して確かめる**ためのノートブックである。
中心にあるのは定理3.2（三つの定式化の同値性）で、分散最大化・再構成誤差最小化・低ランク近似が
同じ答えを返すことを数値で見る。あわせて中心化とスケーリングという二つの前処理の落とし穴
（例3.13、例3.14）、白色化と Z-score の違い（図3.5）、確率的 PCA の最尤解（定理3.22）、
そして実データ（`load_digits`）での主成分を扱う。

**データ行列の規約**：講義ノート全体を通じて $\boldsymbol{X}\in\mathbb{R}^{d\times n}$ は
**列がサンプル**である。`scikit-learn` は行がサンプルなので、渡すときに `X.T` と転置する。

## 目次

1. [準備](#setup)
2. [3.1 三つの定式化は同じ答えを返す](#s1)
3. [3.2 中心化を忘れると主軸が平均の方向に引かれる](#s2)
4. [3.3 寄与率・スクリープロット・次元の選び方](#s3)
5. [3.4 スケーリング：共分散行列 PCA と相関行列 PCA](#s4)
6. [3.5 白色化と Z-score は別物である](#s5)
7. [3.6 確率的 PCA（PPCA）の最尤解](#s6)
8. [3.7 実データ：手書き数字の主成分](#s7)
9. [演習](#ex)
10. [演習の解答](#sol)

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

<a name="s1"></a>
## 3.1 三つの定式化は同じ答えを返す

定理3.2 は次の三つが共通の最適解 $\boldsymbol{W}=\boldsymbol{U}_k$ を持つと主張する。

$$\text{(F1)}\ \max_{\boldsymbol{W}^\top\boldsymbol{W}=\boldsymbol{I}_k}\operatorname{tr}(\boldsymbol{W}^\top\widehat{\boldsymbol{\Sigma}}\boldsymbol{W}),\quad
\text{(F2)}\ \min_{\boldsymbol{W}^\top\boldsymbol{W}=\boldsymbol{I}_k}\tfrac1n\|\widetilde{\boldsymbol{X}}-\boldsymbol{W}\boldsymbol{W}^\top\widetilde{\boldsymbol{X}}\|_F^2,\quad
\text{(F3)}\ \min_{\boldsymbol{Z},\,\boldsymbol{W}^\top\boldsymbol{W}=\boldsymbol{I}_k}\|\widetilde{\boldsymbol{X}}-\boldsymbol{W}\boldsymbol{Z}\|_F^2$$

蝶番になるのは補題3.1（ピタゴラスの定理）から出る分解（式(3.5)）
$\operatorname{tr}\widehat{\boldsymbol{\Sigma}}=\operatorname{tr}(\boldsymbol{W}^\top\widehat{\boldsymbol{\Sigma}}\boldsymbol{W})+(\text{残差})$
であり、左辺が $\boldsymbol{W}$ に依存しないので「射影の分散の最大化」と「残差の最小化」が同じ問題になる。

まず例3.8 の $2\times5$ データで、$k=1$ の場合に方向 $\boldsymbol{w}(\theta)=(\cos\theta,\sin\theta)^\top$ を
一周させて両者の山と谷が同じ位置に来ることを見る。

In [ ]:
# 例3.8 のデータ（d=2, n=5、列がサンプル）
X = np.array([[1., 2., 3., 4., 5.],
              [1., 3., 5., 2., 4.]])
d, n = X.shape
Xt = X - X.mean(axis=1, keepdims=True)          # 中心化データ行列 Xtilde = X H
S = Xt @ Xt.T / n                               # 標本共分散（分母 n）
lam, U = np.linalg.eigh(S)
lam, U = lam[::-1], U[:, ::-1]                  # 降順に並べ替える

print("Sigmahat =\n", S)
print("固有値 lambda =", lam)
print("u1 =", U[:, 0], " u2 =", U[:, 1])

# 方向を一周させて (F1) の射影二乗和と (F2) の残差二乗和を計算する
th = np.linspace(0.0, np.pi, 721)
Wth = np.vstack([np.cos(th), np.sin(th)])       # 2 x 721
proj = n * np.einsum("it,ij,jt->t", Wth, S, Wth)   # sum_i (w^T xtilde_i)^2
total = (Xt ** 2).sum()                            # sum_i ||xtilde_i||^2 = n tr(Sigmahat)
resid = total - proj

i_max, i_min = int(np.argmax(proj)), int(np.argmin(resid))
print("\n全分散 sum_i ||xtilde_i||^2 =", total, "= n tr(Sigmahat) =", n * np.trace(S))
print("射影二乗和が最大の角度 =", np.degrees(th[i_max]).round(3), "度",
      " 残差二乗和が最小の角度 =", np.degrees(th[i_min]).round(3), "度")
print("u1 の角度 =", round(np.degrees(np.arctan2(U[1, 0], U[0, 0])), 3), "度")
print("最大の射影二乗和 =", round(proj[i_max], 6), "= n lambda_1 =", n * lam[0])
print("最小の残差二乗和 =", round(resid[i_min], 6), "= n lambda_2 =", n * lam[1])
print("分解 20 = 15 + 5 :", total, "=", round(proj[i_max], 6), "+", round(resid[i_min], 6))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax = axes[0]
ax.scatter(Xt[0], Xt[1], s=60, color=C["blue"], zorder=3, label=L("中心化データ", "centred data"))
for j, (col, sty) in enumerate([(C["red"], "-"), (C["green"], "--")]):
    v = U[:, j] * 3.0
    ax.plot([-v[0], v[0]], [-v[1], v[1]], sty, color=col, lw=2,
            label=L(f"第{j+1}主成分方向", f"PC{j+1} direction"))
ax.set_aspect("equal"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title(L("例3.8 のデータと主軸", "data of Example 3.8"))
ax.legend(fontsize=9)

ax = axes[1]
deg = np.degrees(th)
ax.plot(deg, proj, color=C["blue"], lw=2, label=L("(F1) 射影の二乗和", "(F1) projected sum of squares"))
ax.plot(deg, resid, color=C["red"], lw=2, label=L("(F2) 残差の二乗和", "(F2) residual sum of squares"))
ax.plot(deg, proj + resid, color=C["gray"], ls=":", lw=2, label=L("和（一定）", "sum (constant)"))
ax.axvline(deg[i_max], color=C["green"], ls="--", lw=1.2)
ax.set_xlabel(L("方向の角度（度）", "angle of direction (deg)"))
ax.set_ylabel(L("二乗和", "sum of squares"))
ax.set_title(L("(F1) の最大と (F2) の最小は一致する", "(F1) max and (F2) min coincide"))
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

左の図の $\boldsymbol{u}_1$ は $45^\circ$ 方向、右の図では青（射影の二乗和）の山と赤（残差の二乗和）の谷が
どちらも $45^\circ$ にあり、灰色の点線（両者の和）は $\theta$ によらず一定である。
これが補題3.1・式(3.5) から出る分解 $20=15+5$ の可視化にほかならない。

次に $k>1$、$d>2$ でも三つの定式化が一致することを確かめる。(F3) の最適値は
Eckart--Young の定理から $\sum_{j>k}\sigma_j^2=n\sum_{j>k}\lambda_j$ になるはずである。
また命題3.9 のとおりスコアの標本共分散は $\operatorname{diag}(\lambda_1,\dots,\lambda_k)$ になる。

In [ ]:
rng = np.random.default_rng(0)
d, n, k = 6, 300, 3
A = rng.standard_normal((d, d))
X = A @ rng.standard_normal((d, n))              # X は d x n（列がサンプル）
Xt = X - X.mean(axis=1, keepdims=True)
S = Xt @ Xt.T / n
lam, U = np.linalg.eigh(S); lam, U = lam[::-1], U[:, ::-1]
Uk = U[:, :k]

f1 = np.trace(Uk.T @ S @ Uk)                     # (F1) の値
f2 = np.linalg.norm(Xt - Uk @ Uk.T @ Xt, "fro") ** 2 / n     # (F2) の値
Z = Uk.T @ Xt                                    # スコア（k x n、列が z_i）
f3 = np.linalg.norm(Xt - Uk @ Z, "fro") ** 2     # (F3) の値（Z の最適値 Z = Uk^T Xt を代入）

print("(F1) tr(W^T S W)      =", f1, " 理論値 sum_{j<=k} lambda_j =", lam[:k].sum())
print("(F2) 残差 /n          =", f2, " 理論値 sum_{j>k} lambda_j  =", lam[k:].sum())
print("(F3) ||Xt - W Z||_F^2 =", f3, " 理論値 n sum_{j>k} lambda_j =", n * lam[k:].sum())
print("(F1)+(F2) =", f1 + f2, " = tr(Sigmahat) =", np.trace(S))

# ランダムな正規直交 W と比べる（貪欲に選んだ Uk が本当に最良か）
best = -np.inf
for _ in range(2000):
    Q, _ = np.linalg.qr(rng.standard_normal((d, k)))
    best = max(best, np.trace(Q.T @ S @ Q))
print("\nランダムな W 2000 本での最大 tr(W^T S W) =", round(best, 6),
      "<= Uk での値", round(f1, 6))

print("スコアの標本共分散（対角のはず）=\n", np.round(np.cov(Z, bias=True), 10))
print("対角成分と lambda_1..k の最大差 =", np.abs(np.diag(np.cov(Z, bias=True)) - lam[:k]).max())

三つの値は理論値と倍精度の丸め誤差の範囲で一致し、(F1)+(F2) は $\operatorname{tr}\widehat{\boldsymbol{\Sigma}}$ に等しい。
ランダムな正規直交 $\boldsymbol{W}$ を 2000 本試しても $\boldsymbol{U}_k$ の値を超えないのは Ky Fan の定理（定理3.6）と
系3.7 の主張——一方向ずつ選ぶ貪欲法が $k$ 次元部分空間の大域最適と一致する——の数値的な確認である。
スコアの標本共分散が対角行列になるのは命題3.9 である。

<a name="s2"></a>
## 3.2 中心化を忘れると主軸が平均の方向に引かれる

$\frac1n\boldsymbol{X}\boldsymbol{X}^\top=\widehat{\boldsymbol{\Sigma}}+\bar{\boldsymbol{x}}\bar{\boldsymbol{x}}^\top$（式(3.13)）の第二項は
階数 1 の半正定値行列で、$\|\bar{\boldsymbol{x}}\|$ が大きいほど支配的になる。
例3.13 のデータでは、中心化しない「第 1 主成分」が真の主軸と**ちょうど直交する**。
これを再現し、あわせて演習3.4(2) の Weyl の不等式
$\lambda_1\le\lambda_1'\le\lambda_1+\|\bar{\boldsymbol{x}}\|^2$ も確かめる（図3.3 に対応）。

In [ ]:
# 例3.13 のデータ（d=2, n=5、列がサンプル）
X = np.array([[3., 4., 5., 6., 7.],
              [7., 5., 6., 4., 3.]])
n = X.shape[1]
xbar = X.mean(axis=1)
Xt = X - xbar[:, None]
S = Xt @ Xt.T / n                                # 中心化した標本共分散
M = X @ X.T / n                                  # 中心化しない二次モーメント

def top_eig(A):
    w, V = np.linalg.eigh(A)
    return w[::-1], V[:, ::-1]

lam, U = top_eig(S)
lam2, U2 = top_eig(M)
ang = lambda v: np.degrees(np.arctan2(v[1], v[0])) % 180.0

print("xbar =", xbar, " ||xbar||^2 =", (xbar ** 2).sum())
print("Sigmahat =\n", S, "\n固有値 =", lam, " u1 =", U[:, 0], " 角度 =", round(ang(U[:, 0]), 2), "度")
print("\nX X^T / n =\n", M, "\n固有値 =", lam2, " 最大固有ベクトル =", U2[:, 0],
      " 角度 =", round(ang(U2[:, 0]), 2), "度")
print("\n二つの方向の内積 =", round(float(U[:, 0] @ U2[:, 0]), 12), "（直交）")
print("恒等式 X X^T/n - (Sigmahat + xbar xbar^T) の最大絶対値 =",
      np.abs(M - (S + np.outer(xbar, xbar))).max())
print("Weyl:", round(lam[0], 4), "<=", round(lam2[0], 4), "<=",
      round(lam[0] + (xbar ** 2).sum(), 4))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for ax, (Xd, u, ttl, col) in zip(axes, [
        (Xt, U[:, 0], L("中心化してから PCA", "PCA after centring"), C["blue"]),
        (X, U2[:, 0], L("中心化せずに PCA", "PCA without centring"), C["red"])]):
    ax.scatter(Xd[0], Xd[1], s=60, color=col, zorder=3)
    c = Xd.mean(axis=1) if ttl.startswith(L("中心化してから", "PCA after")) else np.zeros(2)
    centred = ttl.startswith(L("中心化してから", "PCA after"))
    t = np.linspace(-3.2, 3.2, 2) * (1.0 if centred else 2.6)
    ax.plot(c[0] + t * u[0], c[1] + t * u[1], color=C["green"],
            lw=2.4, ls="-" if centred else "--",
            label=L("第1主成分方向", "first PC direction"))
    if not centred:
        ax.arrow(0, 0, xbar[0], xbar[1], color=C["orange"], width=0.12,
                 length_includes_head=True, zorder=2,
                 label=L("平均ベクトル", "mean vector"))
        ax.scatter([0], [0], marker="+", s=140, color="k", zorder=4)
        ax.set_xlim(-1.5, 8.5); ax.set_ylim(-1.5, 8.5)
    ax.set_aspect("equal"); ax.set_title(ttl)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(fontsize=9, loc="upper right")
fig.tight_layout(); plt.show()

左は中心化後の主軸で、データが右下がりに伸びている向き（$135^\circ$、すなわち $(1,-1)^\top/\sqrt2$）を
正しく捉えている。右は中心化しない場合で、最大固有値 $50.2$ に属する固有ベクトルは
$45^\circ$ 方向（平均ベクトル $(5,5)^\top$ の向き）を指し、真の主軸と内積 0、すなわち
$90^\circ$ 取り違えている。恒等式（式(3.13)）と Weyl の不等式 $3.8\le50.2\le53.8$ も上の出力で確認できる。

**中心化は恣意的な操作ではない。** 講義ノート §3.7 のとおり、アフィン部分空間へのあてはめで
オフセット $\boldsymbol{b}$ も最適化すると $\boldsymbol{b}=\bar{\boldsymbol{x}}$ が最適になる。中心化とは最適なオフセットを
先に決めておくことである。

<a name="s3"></a>
## 3.3 寄与率・スクリープロット・次元の選び方

定義3.12 の寄与率 $r_j=\lambda_j/\sum_l\lambda_l$ と累積寄与率 $R_k$ を（図3.2 に対応）、
真の次元がわかっている人工データ（演習3.7 と同じ設定：$d=30$、$n=200$、真の $k=5$）で見る。
$k$ の選び方として §3.6 の二つを比べる（演習3.7 と同じ二つの規準）。

* **Kaiser 基準**：相関行列 PCA で $\lambda_j>1$ なる $j$ まで採る。
* **平行分析**：各変数（＝各**行**）を独立にランダム置換して相関構造を壊したデータを 500 個作り、
  $j$ 番目の固有値の上側 95% 点を超える $j$ まで採る。

演習3.7 と同じくノイズ分散を $1$（同 (1)）と $4$（同 (3)）に変え、さらに $9$ まで上げて、
どちらの規準が真の $k=5$ に近い答えを返すかを見る。

In [ ]:
def make_data(noise_sd, rng):
    """X = A Z + E（X は d x n、真の潜在次元 5）"""
    A = rng.standard_normal((30, 5))
    Zl = rng.standard_normal((5, 200))
    return A @ Zl + noise_sd * rng.standard_normal((30, 200))

def eig_desc(A):
    return np.sort(np.linalg.eigvalsh(A))[::-1]

def parallel_analysis(X, n_perm=500, seed=1):
    rng = np.random.default_rng(seed)
    out = np.empty((n_perm, X.shape[0]))
    for b in range(n_perm):
        Xp = np.vstack([rng.permutation(row) for row in X])   # 行（変数）ごとに置換
        out[b] = eig_desc(np.corrcoef(Xp))
    return np.percentile(out, 95, axis=0)

res, fig_data = {}, {}
for sd in (1.0, 2.0, 3.0):
    X = make_data(sd, np.random.default_rng(0))               # X は d x n
    lam = eig_desc(np.corrcoef(X))                            # 相関行列 PCA
    thr = parallel_analysis(X, 500)
    k_kaiser = int((lam > 1).sum())
    k_pa = int(np.argmax(lam < thr))                          # 最初に下回る位置
    res[sd] = (k_kaiser, k_pa)
    fig_data[sd] = (lam, thr)
    print(f"ノイズ分散 {sd**2:>4.1f}: 寄与率上位5 =", np.round(lam[:5] / lam.sum(), 4),
          f" 累積 R_5 = {lam[:5].sum()/lam.sum():.4f}")
    print(f"              Kaiser 基準 k = {k_kaiser},  平行分析 k = {k_pa}  （真の k = 5）")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0), sharey=True)
for ax, sd in zip(axes, (1.0, 2.0, 3.0)):
    lam, thr = fig_data[sd]
    j = np.arange(1, 31)
    ax.bar(j, lam, color=C["blue"], alpha=0.75, label=L("固有値", "eigenvalue"))
    ax.plot(j, thr, "o-", ms=3.5, color=C["red"], lw=1.6,
            label=L("平行分析の95%点", "parallel analysis 95%"))
    ax.axhline(1.0, color=C["green"], ls="--", lw=1.4, label=L("Kaiser 基準", "Kaiser criterion"))
    ax.axvline(5.5, color=C["gray"], ls=":", lw=1.4, label=L("真の k=5", "true k=5"))
    ax.set_xlabel(L("主成分の番号 $j$", "component index $j$"))
    ax.set_title(L(f"ノイズ分散 {sd**2:.0f}", f"noise variance {sd**2:.0f}"))
axes[0].set_ylabel(L("相関行列の固有値", "eigenvalue of correlation matrix"))
axes[0].legend(fontsize=9)
fig.tight_layout(); plt.show()

信号が強いとき（ノイズ分散 1）はどちらの規準も真の $k=5$ を返す。
ノイズ分散を上げると Kaiser 基準は $k=6$（分散 4）、$k=9$（分散 9）と採りすぎるのに対し、
平行分析は三つの水準すべてで $k=5$ を保つ。
$d=30$、$n=200$ ではノイズだけでも 1 を超える固有値が多数現れるからで、閾値 1 は
「母相関行列が単位行列」という理想状況の値にすぎない。平行分析は有限標本の揺らぎを
直接シミュレートするので閾値が 1 より上に来て（図の赤い折れ線）、過大評価が是正される。

なお定理3.10（式(3.10)）が示すとおり、累積寄与率 $R_k$ は「対距離の二乗和の保存率」でもある
（演習1 で確かめる）。

<a name="s4"></a>
## 3.4 スケーリング：共分散行列 PCA と相関行列 PCA

例3.14 は、身長を cm から mm に変えるだけで共分散行列 PCA の第 1 主成分が大きく傾くことを示す。
命題3.15 のとおり PCA は直交変換には同変だが、$\boldsymbol{A}=\operatorname{diag}(10,1)$ のような
伸縮には同変でない。相関行列 PCA（＝各変数を標準偏差で割ってから PCA）なら単位に依存しない
（図3.4 に対応）。

In [ ]:
rng = np.random.default_rng(0)
n = 100
Lmat = np.array([[6.0, 0.0], [0.7 * 5, 5 * np.sqrt(1 - 0.7 ** 2)]])
X_cm = Lmat @ rng.standard_normal((2, n)) + np.array([[170.0], [65.0]])   # d=2 x n=100
X_mm = X_cm.copy(); X_mm[0] *= 10.0                                      # 第1変数を mm に

def pca_dir(X, use_corr=False):
    """固有値、解析を行った空間での第1主成分方向、元の単位での向き、標準偏差を返す。"""
    Xt = X - X.mean(axis=1, keepdims=True)
    S = Xt @ Xt.T / X.shape[1]
    s = np.sqrt(np.diag(S))
    A = S / np.outer(s, s) if use_corr else S      # 相関行列 = D^{-1/2} S D^{-1/2}
    w, V = np.linalg.eigh(A); w, V = w[::-1], V[:, ::-1]
    u = V[:, 0] * np.sign(V[0, 0])                 # 解析空間での方向
    u_orig = u * s if use_corr else u              # 元の単位に戻した向き
    return w, u, u_orig / np.linalg.norm(u_orig), s

for name, X in [("cm", X_cm), ("mm", X_mm)]:
    for label, uc in [(L("共分散行列", "covariance"), False), (L("相関行列", "correlation"), True)]:
        w, u, u_orig, s = pca_dir(X, uc)
        print(f"{name:>2} / {label}: 解析空間での角度 = {np.degrees(np.arctan2(u[1], u[0])):7.3f} 度"
              f"  寄与率 = {w[0]/w.sum():.4f}  固有値 = {np.round(w, 4)}")

# 相関行列そのものは単位変換で不変
print("\n相関行列 (cm) =\n", np.round(np.corrcoef(X_cm), 6))
print("相関行列 (mm) との最大差 =", np.abs(np.corrcoef(X_cm) - np.corrcoef(X_mm)).max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (name, X, unit) in zip(axes, [("cm", X_cm, L("身長 (cm)", "height (cm)")),
                                      ("mm", X_mm, L("身長 (mm)", "height (mm)"))]):
    c = X.mean(axis=1)
    ax.scatter(X[0], X[1], s=18, color=C["gray"], alpha=0.8)
    for uc, col, lab in [(False, C["red"], L("共分散行列 PCA", "covariance PCA")),
                         (True, C["blue"], L("相関行列 PCA", "correlation PCA"))]:
        _, _, u_orig, s = pca_dir(X, uc)
        v = 2.2 * u_orig * np.linalg.norm(s)       # 元の単位での向きを描く
        ax.plot([c[0] - v[0], c[0] + v[0]], [c[1] - v[1], c[1] + v[1]], color=col, lw=2.2, label=lab)
    ax.set_xlabel(unit); ax.set_ylabel(L("体重 (kg)", "weight (kg)"))
    ax.set_title(L(f"単位 = {name}", f"unit = {name}")); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

共分散行列 PCA の第 1 主成分は cm では $38.775^\circ$（寄与率 0.8651）、mm では $3.538^\circ$
（寄与率 0.9966）で、物理的に同じデータなのに向きも寄与率も大きく変わる。$35^\circ$ の差である。
相関行列 PCA は cm でも mm でも標準化空間で $45^\circ$、固有値 $(1.7221,0.2779)$、寄与率 0.8611 と
完全に同じ答えを返す。相関行列そのものが単位変換で不変（上の出力で最大差 $1.1\times10^{-16}$）だからである。
図では相関行列 PCA の向きを元の単位に戻して描いているので、軸の縮尺が変われば線の見かけの傾きも変わるが、
「どの標準化変数をどれだけ使うか」という答えは変わっていない。

ただし相関行列 PCA が常に正しいわけではない。すべての変数が同じ単位で測られ分散の大小自体が
意味を持つ場合（画像の画素値など）は共分散行列 PCA が自然で、無理に標準化すると
ほとんど動かないノイズ変数が過大に重みづけられる（§3.7）。

<a name="s5"></a>
## 3.5 白色化と Z-score は別物である

定義3.16 の PCA 白色化 $\boldsymbol{Z}_{\mathrm{white}}=\boldsymbol{\Lambda}^{-1/2}\boldsymbol{U}^\top\widetilde{\boldsymbol{X}}$ は
共分散を $\boldsymbol{I}$ にする（命題3.17）。いっぽう実務でよく使われる Z-score
$\boldsymbol{Z}_{\mathrm{std}}=\boldsymbol{D}^{-1/2}\widetilde{\boldsymbol{X}}$ は式(3.14)のとおり共分散を**相関行列**にするだけで、
非対角成分は相関係数のまま残る。図3.5 の 4 つの変換を再現する。
あわせて命題3.18（ZCA の最適性）——白色化のうち元データに最も近いのは
$\boldsymbol{A}=\widehat{\boldsymbol{\Sigma}}^{-1/2}$ ——も数値で確かめる。

In [ ]:
rng = np.random.default_rng(0)
n = 400
Lmat = np.array([[1.0, 0.0], [1.6, 0.6]])
X = Lmat @ rng.standard_normal((2, n))            # d=2 x n=400
Xt = X - X.mean(axis=1, keepdims=True)
S = Xt @ Xt.T / n
lam, U = np.linalg.eigh(S); lam, U = lam[::-1], U[:, ::-1]

Z_rot = U.T @ Xt                                  # PCA 回転
Z_pca = np.diag(lam ** -0.5) @ U.T @ Xt           # PCA 白色化
Z_zca = U @ Z_pca                                 # ZCA 白色化（= Sigmahat^{-1/2} Xt）
Z_std = Xt / np.sqrt(np.diag(S))[:, None]         # Z-score（変数ごとの標準化）

def corr_of(Z):
    Sz = Z @ Z.T / Z.shape[1]
    return Sz, Sz[0, 1] / np.sqrt(Sz[0, 0] * Sz[1, 1])

for name, Z in [(L("元データ", "original"), Xt), (L("PCA 回転", "PCA rotation"), Z_rot),
                (L("PCA 白色化", "PCA whitening"), Z_pca), ("Z-score", Z_std),
                (L("ZCA 白色化", "ZCA whitening"), Z_zca)]:
    Sz, r = corr_of(Z)
    print(f"{name:<12} 分散 = {np.round(np.diag(Sz), 4)}  相関 = {r:+.4f}")

print("\n白色化の検証 max|ZZ^T/n - I|: PCA =",
      f"{np.abs(Z_pca @ Z_pca.T / n - np.eye(2)).max():.2e}",
      " ZCA =", f"{np.abs(Z_zca @ Z_zca.T / n - np.eye(2)).max():.2e}")
print("元データとの距離 ||A Xt - Xt||_F : PCA 白色化 =", round(np.linalg.norm(Z_pca - Xt), 4),
      " ZCA 白色化 =", round(np.linalg.norm(Z_zca - Xt), 4), "（命題3.18：ZCA が最小）")

ang0 = np.arctan2(Xt[1], Xt[0])                   # 点の色は「元データでの角度」
fig, axes = plt.subplots(2, 4, figsize=(13, 6.2))
panels = [(Xt, L("(a) 元データ", "(a) original")), (Z_rot, L("(b) PCA 回転", "(b) PCA rotation")),
          (Z_pca, L("(c) PCA 白色化", "(c) PCA whitening")), (Z_std, L("(d) Z-score", "(d) Z-score"))]
for j, (Z, ttl) in enumerate(panels):
    ax = axes[0, j]
    ax.scatter(Z[0], Z[1], c=ang0, cmap="twilight", s=10)
    ax.set_aspect("equal"); ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
    Sz, r = corr_of(Z)
    ax = axes[1, j]
    im = ax.imshow(Sz, cmap="RdBu_r", vmin=-2.6, vmax=2.6)
    for a in range(2):
        for b in range(2):
            ax.text(b, a, f"{Sz[a, b]:.3f}", ha="center", va="center", fontsize=10)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.grid(False)
    ax.set_title(L(f"共分散（相関 {r:+.3f}）", f"covariance (corr {r:+.3f})"), fontsize=10)
fig.tight_layout(); plt.show()

上段が散布図、下段が対応する共分散行列である。点の色は**元データでの角度**なので、
同じ色を追えば各変換での移動先がわかる。

* (b) PCA 回転は雲を軸に揃えるだけで、分散は $\lambda_1,\lambda_2$ のまま。
* (c) PCA 白色化はさらに各軸を標準偏差で割るので雲が等方的になり、共分散が $\boldsymbol{I}$、相関は 0。
* (d) Z-score は対角を 1 にするが**相関は元のまま残り**、雲は傾いたままである。

$\widehat{\boldsymbol{\Sigma}}$ が対角行列のときに限って両者は一致する（式(3.14)）。
また元データとの距離 $\|\boldsymbol{A}\widetilde{\boldsymbol{X}}-\widetilde{\boldsymbol{X}}\|_F$ は
ZCA 白色化 $24.2597$ 対 PCA 白色化 $39.2173$ で ZCA のほうが小さく、命題3.18 の主張どおりである。
なお白色化は $\lambda_j^{-1/2}$ を掛けるので、小さい固有値の方向（＝ほとんどノイズ）を
桁違いに増幅する点は §3.8 の警告のとおり注意が必要である。

<a name="s6"></a>
## 3.6 確率的 PCA（PPCA）の最尤解

定義3.20 の PPCA $\boldsymbol{x}=\boldsymbol{W}\boldsymbol{z}+\boldsymbol{\mu}+\boldsymbol{\varepsilon}$
（$\boldsymbol{z}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}_k)$, $\boldsymbol{\varepsilon}\sim\mathcal{N}(\boldsymbol{0},\sigma^2\boldsymbol{I}_d)$）の
最尤解は定理3.22（Tipping--Bishop）で、対数尤度 式(3.16) の大域最大として

$$\hat\sigma^2=\frac{1}{d-k}\sum_{j>k}\lambda_j,\qquad
\widehat{\boldsymbol{W}}=\boldsymbol{U}_k(\boldsymbol{\Lambda}_k-\hat\sigma^2\boldsymbol{I}_k)^{1/2}\boldsymbol{Q}$$

と閉じた形で与えられる（式(3.17)、式(3.19)）。$\hat\sigma^2$ は**捨てられた方向の平均分散**である。
固有分解から直接この式を計算し、(i) `scikit-learn` の `PCA` が返す `noise_variance_` と一致すること、
(ii) 対数尤度がランダム摂動で改善されないこと（すなわち停留点であること）を確かめる。

`scikit-learn` の `explained_variance_` は分母 $n-1$ の流儀なので、比較用にこちらも $n-1$ で計算する。

In [ ]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
d, k, n, sigma2_true = 5, 2, 400, 0.16
Wtrue = rng.standard_normal((d, k))
mu = np.array([3.0, -1.0, 0.5, 2.0, 0.0])
X = (Wtrue @ rng.standard_normal((k, n))
     + np.sqrt(sigma2_true) * rng.standard_normal((d, n)) + mu[:, None])   # d x n

Xt = X - X.mean(axis=1, keepdims=True)
S = Xt @ Xt.T / (n - 1)                       # sklearn と同じ分母 n-1
lam, U = np.linalg.eigh(S); lam, U = lam[::-1], U[:, ::-1]

sig2 = lam[k:].mean()                         # 定理3.22: 捨てた方向の平均分散
What = U[:, :k] * np.sqrt(lam[:k] - sig2)     # W = U_k (Lambda_k - sig2 I)^{1/2}（Q = I）

def loglik(W, s2, S, n, d):
    Cmat = W @ W.T + s2 * np.eye(d)
    sign, logdet = np.linalg.slogdet(Cmat)
    return -0.5 * n * (d * np.log(2 * np.pi) + logdet + np.trace(np.linalg.solve(Cmat, S)))

ll = loglik(What, sig2, S, n, d)
pca = PCA(n_components=k).fit(X.T)            # sklearn は n x d 規約なので転置して渡す
print("固有値 lambda           =", np.round(lam, 6))
print("sigma^2 hat（式より）   =", round(sig2, 6), " 真の値 =", sigma2_true)
print("sklearn noise_variance_ =", round(float(pca.noise_variance_), 6),
      " 差 =", abs(sig2 - pca.noise_variance_))
print("対数尤度 ell（分母 n-1 の S）=", round(ll, 4))
Sn = Xt @ Xt.T / n                            # 分母 n の標本共分散
print("対数尤度 ell（分母 n の S） =", round(loglik(What, sig2, Sn, n, d), 4),
      " sklearn score * n =", round(pca.score(X.T) * n, 4))

# 摂動で改善するか（停留点かどうか）
n_better = 0
for _ in range(200):
    Wp = What + 0.05 * rng.standard_normal(What.shape)
    s2p = sig2 * np.exp(0.05 * rng.standard_normal())
    if loglik(Wp, s2p, S, n, d) > ll:
        n_better += 1
print("標準偏差 0.05 の摂動 200 回のうち尤度が改善した回数 =", n_better)

# sigma^2 -> 0 の極限では事後平均が白色化スコアになる（注意3.24）
Minv = np.linalg.inv(What.T @ What + sig2 * np.eye(k))
post = Minv @ What.T @ Xt                                    # E[z|x]
white = np.diag(lam[:k] ** -0.5) @ U[:, :k].T @ Xt           # 白色化スコア
print("事後平均と白色化スコアの相関 =",
      np.round([np.corrcoef(post[j], white[j])[0, 1] for j in range(k)], 6))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax = axes[0]
ax.bar(np.arange(1, d + 1), lam, color=C["blue"], alpha=0.8, label=L("固有値", "eigenvalue"))
ax.axhline(sig2, color=C["red"], ls="--", lw=1.8,
           label=L(r"$\hat\sigma^2$（捨てた方向の平均）", r"$\hat\sigma^2$ (mean of tail)"))
ax.axvline(k + 0.5, color=C["gray"], ls=":", lw=1.4, label=L(f"k = {k}", f"k = {k}"))
ax.set_xlabel(L("主成分の番号", "component index")); ax.set_ylabel(L("固有値", "eigenvalue"))
ax.set_title(L("PPCA の最尤解", "MLE of PPCA")); ax.legend(fontsize=9)

ax = axes[1]
ks = np.arange(1, d)
lls, bics = [], []
for kk in ks:
    s2k = lam[kk:].mean()
    Wk = U[:, :kk] * np.sqrt(np.maximum(lam[:kk] - s2k, 1e-12))
    lls.append(loglik(Wk, s2k, Sn, n, d))
    n_par = d * kk - kk * (kk - 1) / 2 + 1 + d     # W の自由度 + sigma^2 + mu
    bics.append(-2 * lls[-1] + n_par * np.log(n))
ax.plot(ks, lls, "o-", color=C["purple"], lw=1.8, label=L("対数尤度", "log-likelihood"))
ax.set_xlabel(L("潜在次元 k", "latent dimension k")); ax.set_ylabel(L("対数尤度", "log-likelihood"))
ax2 = ax.twinx()
ax2.plot(ks, bics, "s--", color=C["orange"], lw=1.8, label="BIC"); ax2.grid(False)
ax2.set_ylabel("BIC", color=C["orange"])
ax.axvline(k, color=C["gray"], ls=":", lw=1.4)
ax.set_title(L("k の選択：尤度は単調、BIC は最小を持つ",
               "choosing k: likelihood is monotone, BIC has a minimum"))
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout(); plt.show()
print("k ごとの対数尤度 =", np.round(lls, 2))
print("k ごとの BIC     =", np.round(bics, 2), " 最小は k =", int(ks[int(np.argmin(bics))]))

`noise_variance_` は式 $\hat\sigma^2=\frac{1}{d-k}\sum_{j>k}\lambda_j$ の値と一致し、
`scikit-learn` の `PCA` が（第 $k$ 成分までを取ったとき）まさに PPCA の最尤解を返していることがわかる。
摂動 200 回のうち尤度を改善したものは 0 件で、$(\widehat{\boldsymbol{W}},\hat\sigma^2)$ が停留点であることが
数値的に確かめられる。対数尤度の値は分母の流儀に敏感で、$\hat\sigma^2$ と $\widehat{\boldsymbol{W}}$ を
$n-1$ 版の $\widehat{\boldsymbol{\Sigma}}$ から作りつつ尤度の $\operatorname{tr}(\boldsymbol{C}^{-1}\boldsymbol{S})$ を
分母 $n$ の $\boldsymbol{S}$ で計算すると `scikit-learn` の `score` と完全に一致する（$-2046.4775$）。

右の図は $k$ ごとの対数尤度と BIC である。対数尤度は $k$ について単調に増えるので尤度だけでは
$k$ を選べないが、BIC は $k=2$（＝真の潜在次元）で最小になる。確率モデル化するとこうした
モデル選択の議論ができるようになる（注意3.24）。また $\sigma^2\to0$ の極限で事後平均 $\mathbb{E}[\boldsymbol{z}\mid\boldsymbol{x}]$ が
白色化された主成分スコアに一致するという主張も、両者の相関が 1 になることで確認できる。

<a name="s7"></a>
## 3.7 実データ：手書き数字の主成分

`load_digits`（$8\times8$ の手書き数字、$d=64$、$n=1797$）で主成分を見る。
§3.1 の問い——観測された $d$ 個の変数は本当に $d$ 次元分の情報を持っているのか——を実データで確かめる。
スクリープロット、第 1・第 2 主成分による散布図、そして
定理3.2 の (F2) の意味での再構成（$k$ 本で近似した画像）を並べる。

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data.T                      # sklearn は n x d なので転置して d x n にする
y = digits.target
d, n = X.shape
Xt = X - X.mean(axis=1, keepdims=True)
U, sv, Vt = np.linalg.svd(Xt, full_matrices=False)     # Xtilde = U S V^T
lam = sv ** 2 / n                                      # lambda_j = sigma_j^2 / n
ratio = lam / lam.sum()
cum = np.cumsum(ratio)
k90 = int(np.searchsorted(cum, 0.90) + 1)

print("d =", d, " n =", n)
print("寄与率 上位5 =", np.round(ratio[:5], 4))
print("累積寄与率 R_2 =", round(cum[1], 4), " R_10 =", round(cum[9], 4),
      " R_20 =", round(cum[19], 4))
print("累積寄与率が 90% を超える最小の k =", k90, " (R_k =", round(cum[k90 - 1], 4), ")")

Z = U.T @ Xt                                           # 主成分スコア（列がサンプル）
ks = [2, 8, 16, 32]
print("再構成の相対誤差 1 - R_k :", {kk: round(1 - cum[kk - 1], 4) for kk in ks})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.bar(np.arange(1, 31), ratio[:30], color=C["blue"], alpha=0.8)
ax.set_xlabel(L("主成分の番号 $j$", "component index $j$"))
ax.set_ylabel(L("寄与率 $r_j$", "explained variance ratio $r_j$"))
ax.set_title(L("スクリープロット（上位30）", "scree plot (top 30)"))
ax2 = ax.twinx(); ax2.plot(np.arange(1, 31), cum[:30], "o-", ms=3.5, color=C["red"], lw=1.6)
ax2.axhline(0.9, color=C["gray"], ls=":", lw=1.2); ax2.set_ylim(0, 1.02); ax2.grid(False)
ax2.set_ylabel(L("累積寄与率 $R_k$", "cumulative $R_k$"), color=C["red"])

ax = axes[1]
sc = ax.scatter(Z[0], Z[1], c=y, cmap="tab10", s=8, alpha=0.85)
ax.set_xlabel(L("第1主成分スコア", "PC1 score")); ax.set_ylabel(L("第2主成分スコア", "PC2 score"))
ax.set_title(L("上位2主成分への射影", "projection onto top 2 PCs"))
fig.colorbar(sc, ax=ax, ticks=range(10), label=L("数字", "digit"))
fig.tight_layout(); plt.show()

# 再構成（(F2) の見方：k 次元部分空間への直交射影）
i0 = 17
fig, axes = plt.subplots(1, len(ks) + 1, figsize=(11, 2.6))
axes[0].imshow(X[:, i0].reshape(8, 8), cmap="gray_r"); axes[0].set_title(L("元画像", "original"))
for ax, kk in zip(axes[1:], ks):
    rec = X.mean(axis=1) + U[:, :kk] @ (U[:, :kk].T @ Xt[:, i0])
    ax.imshow(rec.reshape(8, 8), cmap="gray_r")
    ax.set_title(L(f"k={kk} ({cum[kk-1]*100:.0f}%)", f"k={kk} ({cum[kk-1]*100:.0f}%)"), fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.tight_layout(); plt.show()

# 上位主成分方向そのもの（負荷ベクトル u_j は R^64 の元なので画像として見られる）
fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for j, ax in enumerate(axes):
    ax.imshow(U[:, j].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(L(f"$u_{{{j+1}}}$ ({ratio[j]*100:.1f}%)", f"$u_{{{j+1}}}$ ({ratio[j]*100:.1f}%)"),
                 fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.tight_layout(); plt.show()

$d=64$ の画素値のうち、上位 2 主成分で分散の約 28%、上位 21 本で 90% を説明できる。
散布図では 0 と 6、あるいは 4 と 7 のようにいくつかの数字が分かれて見えるが、
2 次元では重なる組も多い。PCA はラベルを一切見ない教師なし手法なので、
「寄与率が高い方向」が「分類に有用な方向」である保証はない（§3.6 の警告）。

再構成画像は $k$ を増やすと元画像に近づく。相対誤差はちょうど $1-R_k$ に等しい
（定理3.2 の (F2)）。負荷ベクトル $\boldsymbol{u}_j$ 自身も $\mathbb{R}^{64}$ の元なので画像として表示でき、
上位の主成分が「太さ」「傾き」といった大域的な濃淡パターンを表していることが見える。

<a name="ex"></a>
## 演習

`# TODO` を埋めて実行せよ。解答は次節にある。

**演習 1（定理3.10：PCA 射影の非拡大性）** $d=8$, $n=60$ の乱数データで、
(1) $k=1,2,3,5$ について対 $(i,j)$ すべてで式(3.9) の $\|\boldsymbol{z}_i-\boldsymbol{z}_j\|\le\|\tilde{\boldsymbol{x}}_i-\tilde{\boldsymbol{x}}_j\|$ を確認し、
(2) 対距離の二乗和の比が累積寄与率 $R_k$ に一致すること（式(3.10)）を確認し、
(3) $k=2$ での個々の比の最小・中央値・最大を求めて、$\sqrt{R_k}$ の付近に集中する**わけではない**ことを見よ。

**演習 2（寄与率が高い $\ne$ 役に立つ、§3.6 の warnbox）** §3.6 の例を実装する。
$x_1\sim\mathcal{N}(0,10^2)$（ラベルと独立）、$x_2\sim\mathcal{N}(2y,0.5^2)$（$y=\pm1$）とし、
第 1 主成分の寄与率と、第 1／第 2 主成分スコアだけを使った分類の正解率を比べよ。

**演習 3（演習3.5(2)、命題3.18：ZCA 白色化行列の手計算）**
$\widehat{\boldsymbol{\Sigma}}=\begin{pmatrix}4&2\\2&4\end{pmatrix}$ の $\widehat{\boldsymbol{\Sigma}}^{-1/2}$ を
固有分解から計算し、$\widehat{\boldsymbol{\Sigma}}^{-1/2}\widehat{\boldsymbol{\Sigma}}\widehat{\boldsymbol{\Sigma}}^{-1/2}=\boldsymbol{I}_2$ を確かめよ。

In [ ]:
# ---- 演習 1 ----
rng = np.random.default_rng(0)
A = rng.standard_normal((8, 8))
X = A @ rng.standard_normal((8, 60))          # d=8 x n=60
Xt = X - X.mean(axis=1, keepdims=True)
S = Xt @ Xt.T / X.shape[1]
lam, U = np.linalg.eigh(S); lam, U = lam[::-1], U[:, ::-1]

def pair_dists(Y):
    """列がサンプルの行列 Y から対距離の配列を返す"""
    D = np.linalg.norm(Y[:, :, None] - Y[:, None, :], axis=0)
    iu = np.triu_indices(Y.shape[1], 1)
    return D[iu]

d_orig = pair_dists(Xt)
for k in (1, 2, 3, 5):
    Z = U[:, :k].T @ Xt
    d_proj = pair_dists(Z)
    n_viol = None     # TODO: 非拡大性 ||z_i - z_j|| <= ||xt_i - xt_j|| に違反する対の数
    ratio2 = None     # TODO: 対距離の二乗和の比
    Rk = None         # TODO: 累積寄与率（lam から作る）
    print(f"k={k}: 違反 {n_viol} 対,  二乗和比 = {ratio2},  累積寄与率 = {Rk}")

# ---- 演習 2 ----
rng = np.random.default_rng(1)
n = 400
y = rng.choice([-1.0, 1.0], size=n)
X2 = np.vstack([10.0 * rng.standard_normal(n), 2.0 * y + 0.5 * rng.standard_normal(n)])  # 2 x n
# TODO: X2 の共分散行列 PCA を行い、寄与率と、各主成分スコアの符号で分類したときの正解率を出す

# ---- 演習 3 ----
Sig = np.array([[4.0, 2.0], [2.0, 4.0]])
# TODO: Sig^{-1/2} を固有分解から作り、Sig^{-1/2} Sig Sig^{-1/2} = I を確認する

<a name="sol"></a>
## 演習の解答

In [ ]:
# ---- 演習 1 の解答 ----
def pair_dists(Y):
    D = np.linalg.norm(Y[:, :, None] - Y[:, None, :], axis=0)
    return D[np.triu_indices(Y.shape[1], 1)]

rng = np.random.default_rng(0)
A = rng.standard_normal((8, 8))
X = A @ rng.standard_normal((8, 60))
Xt = X - X.mean(axis=1, keepdims=True)
S = Xt @ Xt.T / X.shape[1]
lam, U = np.linalg.eigh(S); lam, U = lam[::-1], U[:, ::-1]
d_orig = pair_dists(Xt)

print("[演習1]")
for k in (1, 2, 3, 5):
    Z = U[:, :k].T @ Xt
    d_proj = pair_dists(Z)
    n_viol = int((d_proj > d_orig + 1e-12).sum())
    ratio2 = (d_proj ** 2).sum() / (d_orig ** 2).sum()
    Rk = lam[:k].sum() / lam.sum()
    print(f"  k={k}: 違反 {n_viol} 対,  二乗和比 = {ratio2:.6f},  累積寄与率 = {Rk:.6f}"
          f",  差 = {abs(ratio2 - Rk):.2e}")

k = 2
r_ind = pair_dists(U[:, :k].T @ Xt) / d_orig
Rk = lam[:k].sum() / lam.sum()
print(f"  k=2 の個々の比: 最小 {r_ind.min():.4f}, 中央値 {np.median(r_ind):.4f}, "
      f"最大 {r_ind.max():.4f}   sqrt(R_2) = {np.sqrt(Rk):.4f}")
print(f"  比が 0.9 を超える対 {int((r_ind > 0.9).sum())}/{r_ind.size}, "
      f"0.5 未満の対 {int((r_ind < 0.5).sum())}/{r_ind.size}")

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.hist(r_ind, bins=40, color=C["blue"], alpha=0.85)
ax.axvline(np.sqrt(Rk), color=C["red"], ls="--", lw=2,
           label=L(r"$\sqrt{R_2}$", r"$\sqrt{R_2}$"))
ax.set_xlabel(L("個々の距離の比", "ratio of individual distances"))
ax.set_ylabel(L("対の数", "number of pairs"))
ax.set_title(L("式(3.10) は総和の等式であって個々の比ではない",
               "eq. (3.9) holds for the sum, not for each pair"))
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

# ---- 演習 2 の解答 ----
rng = np.random.default_rng(1)
n = 400
y = rng.choice([-1.0, 1.0], size=n)
X2 = np.vstack([10.0 * rng.standard_normal(n), 2.0 * y + 0.5 * rng.standard_normal(n)])
Xt2 = X2 - X2.mean(axis=1, keepdims=True)
S2 = Xt2 @ Xt2.T / n
lam2, U2 = np.linalg.eigh(S2); lam2, U2 = lam2[::-1], U2[:, ::-1]
Z2 = U2.T @ Xt2
print("\n[演習2]")
print("  標本共分散 =\n", np.round(S2, 3))
print("  寄与率 =", np.round(lam2 / lam2.sum(), 4))
for j in range(2):
    acc = max((np.sign(Z2[j]) == y).mean(), (np.sign(-Z2[j]) == y).mean())
    print(f"  第{j+1}主成分スコアの符号による正解率 = {acc:.3f}"
          f"（寄与率 {lam2[j]/lam2.sum():.4f}）")
print("  → 寄与率 96% の第1主成分は分類に使えず、寄与率 4% の第2主成分だけが使える")

# ---- 演習 3 の解答 ----
Sig = np.array([[4.0, 2.0], [2.0, 4.0]])
w, V = np.linalg.eigh(Sig); w, V = w[::-1], V[:, ::-1]
Sig_inv_half = V @ np.diag(w ** -0.5) @ V.T
print("\n[演習3]")
print("  固有値 =", w, " 固有ベクトル =\n", np.round(V, 6))
print("  Sigmahat^{-1/2} =\n", np.round(Sig_inv_half, 5))
print("  検算 max|Sig^{-1/2} Sig Sig^{-1/2} - I| =",
      f"{np.abs(Sig_inv_half @ Sig @ Sig_inv_half - np.eye(2)).max():.2e}")